### EC-NAS Feature Engineering

Builds the shared-schema feature table for EC-NAS (CNN family), matching `02a_butter_e_features.ipynb`'s column names for pooling later.

**Source data and format.** EC-NAS-Bench (https://github.com/saintslab/EC-NAS-Bench) ships its energy benchmark as TFRecord files requiring TensorFlow 1.x-compat and Google's vendored `nasbench` protobuf classes to decode — none of which were available or worth installing. Instead, the same repo also ships the *raw, per-run* JSON result files underneath the TFRecords (`ecnas/vendors/ec_nasbench/nasbench/data/train_model_results/energy/...`), which are plain and directly parseable. Only that subdirectory plus the architecture-graph definitions (`.../nasbench/data/graphs/generated_graphs_{4V9E,5V9E}.json`) were pulled from the repo via a sparse git checkout into `data/raw/ec_nas/`.

**Actual scope of measured data — smaller than the ~423,000-architecture figure in the project notes.** That figure is NAS-Bench-101's *accuracy* benchmark. The *energy*-measured subset is much narrower:

| Vertex group | Unique architectures | Repeats | Runs |
|---|---|---|---|
| 4V9E | 91 | 3 | 273 |
| 5V9E | 2,532 | 1 | 2,532 |
| **Total** | **2,623** | | **2,805** |

All of it measured at a fixed **4 epochs** — 7V9E and the 12/36/108-epoch variants exist only as `surrogate` (MLP-predicted) or `linscale` (linearly-extrapolated) TFRecords, i.e. not real measurements, so they're excluded. `epochs` is therefore a documented constant `4` here, same treatment as BUTTER-E's constant `3000` — just a different fixed budget for a different, smaller, real subset.

**Column mapping decisions:**
- `params` — `trainable_params` from each `results.json`, EC-NAS's own reported parameter count (their |θ|)
- `depth` — number of vertices `|V|` in the architecture's DAG (the adjacency matrix dimension from `generated_graphs_*.json`), the closest available equivalent to BUTTER-E's literal MLP layer depth — documented as an analog, not an exact match
- `flops` — no FLOPs field exists in EC-NAS; same `2 × params` approximation as BUTTER-E, for consistency
- `epochs` — fixed constant `4` (see above)
- `batch_size` — fixed constant `256`, confirmed directly from the vendored NAS-Bench-101 source (`nasbench/lib/config.py`: `flags.DEFINE_integer("batch_size", 256, ...)`, "the default values are exactly what is used during the exhaustive evaluation of all models"). Not architecture-varying, so this is a real value, not a fabrication.
- `target` — `total_energy (kWh)`, the whole-run measurement (parallel to BUTTER-E's raw `energy`, not its idle-corrected `std_energy` — EC-NAS has no idle-power correction available), converted to joules (`× 3,600,000`) to match BUTTER-E's units
- `family` = `"CNN"`, `source_dataset` = `"EC-NAS"` (constants)
- `run_id` = `"{vertex_group}_{architecture_hash}_{repeat}"`, unique per training run, playing the same traceability role as BUTTER-E's `run_id`; `architecture_hash` is also kept as its own column since it identifies the architecture across repeats (91 hashes recur across both vertex groups as genuinely separate measurements — see note below)

Hardware-utilization / arithmetic-intensity feature, and the 4V hardware-specific multi-GPU sub-benchmark (RQ2 basis) — both deferred; the latter exists only as TFRecord/protobuf and needs the heavier pipeline this notebook deliberately avoided.

In [ ]:
# IMPORTS

import glob
import json

import numpy as np
import pandas as pd

In [ ]:
# LOAD ARCHITECTURE GRAPH DEFINITIONS
# hash -> [adjacency_matrix, encoded_operations]; matrix dimension = |V|, used as the depth analog

DATA_PATH = "../../data/raw/ec_nas/"
VERTEX_GROUPS = ["4V9E", "5V9E"]

graphs = {}
for group in VERTEX_GROUPS:
    with open(DATA_PATH + f"graphs/generated_graphs_{group}.json") as f:
        graphs[group] = json.load(f)

{group: len(g) for group, g in graphs.items()}

In [ ]:
# LOAD PER-RUN RESULTS
# note: 91 architecture hashes appear in *both* the 4V9E and 5V9E batches as genuinely
# separate measurements (different timestamps, different energy readings, different
# regional carbon intensity -- confirmed by diffing two colliding files) -- not duplicates.
# run_id includes the vertex group to keep these distinct.

pattern = DATA_PATH + "train_model_results/energy/*/4_epochs/*/*/repeat_*/results.json"
files = glob.glob(pattern)
print("result files found:", len(files))

rows = []
for fp in files:
    parts = fp.replace("\\", "/").split("/")
    # .../energy/<group>/4_epochs/<hash_bucket>/<hash>/<repeat>/results.json
    group = parts[-6]
    arch_hash = parts[-3]
    repeat = parts[-2]

    with open(fp) as f:
        result = json.load(f)

    matrix, _operations = graphs[group][arch_hash]
    num_vertices = len(matrix)

    rows.append({
        'run_id': f'{group}_{arch_hash}_{repeat}',
        'architecture_hash': arch_hash,
        'vertex_group': group,
        'params': result['trainable_params'],
        'depth': num_vertices,
        'total_energy_kwh': result['total_energy (kWh)'],
    })

raw = pd.DataFrame(rows)
print("duplicate run_ids:", raw['run_id'].duplicated().sum())
raw.shape

In [ ]:
# BUILD SHARED-SCHEMA FEATURE TABLE
# batch_size = 256, fixed: confirmed directly from the vendored NAS-Bench-101 source
# (ecnas/vendors/ec_nasbench/nasbench/lib/config.py), which EC-NAS inherits unmodified --
# `flags.DEFINE_integer("batch_size", 256, "Training batch size.")`, with the comment
# "the default values are exactly what is used during the exhaustive evaluation of all
# models." Not architecture-varying, so safe to set as a constant rather than leaving null.

features = pd.DataFrame({
    'run_id': raw['run_id'],
    'params': raw['params'],
    'depth': raw['depth'],
    'flops': 2 * raw['params'],                        # same approximation as BUTTER-E
    'epochs': 4,                                        # fixed budget, see notebook intro
    'batch_size': 256,                                   # fixed, confirmed from NAS-Bench-101 source
    'target': raw['total_energy_kwh'] * 3_600_000,       # kWh -> joules, matches BUTTER-E's unit
    'family': 'CNN',
    'source_dataset': 'EC-NAS',
})

features.shape

In [ ]:
# SAVE

OUT_PATH = "../../data/processed/ec_nas/ec_nas_features.csv"
features.to_csv(OUT_PATH, index=False)
OUT_PATH

In [ ]:
# SANITY CHECK — same set of prints as 02a, for side-by-side comparison before combining

print(features.shape)
display(features.head())
print()
print(features.dtypes)
print()
print("nulls:\n", features.isnull().sum())